In [0]:
# Bronze: raw Delta -> explode JSON -> MERGE (incremental, idempotent)
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG      = "sample_project_playground_003"
RAW_TABLE    = f"{CATALOG}.raw.entsoe_raw_json"
BRONZE_TABLE = f"{CATALOG}.bronze.entsoe_day_ahead_prices_se"
CKPT         = f"/Volumes/{CATALOG}/raw/checkpoints/entsoe_bronze_prices"

# --- DDL (idempotent) -------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  bidding_zone            STRING,
  area_eic                STRING,
  contract_type           STRING,
  classification_sequence STRING,
  data_item_code          STRING,
  currency                STRING,
  data_provider_org       STRING,
  curve_type              STRING,
  resolution              STRING,
  interval_start_utc      TIMESTAMP,
  interval_end_utc        TIMESTAMP,
  point_index             INT,
  price_eur_mwh           DOUBLE,
  delivery_date_local     DATE,
  source_folder           STRING,
  source_file_path        STRING,
  bronze_ingested_at      TIMESTAMP
)
USING DELTA
CLUSTER BY (bidding_zone)
""")

COLS = ["bidding_zone", "area_eic", "contract_type", "classification_sequence",
        "data_item_code", "currency", "data_provider_org", "curve_type", "resolution",
        "interval_start_utc", "interval_end_utc", "point_index", "price_eur_mwh",
        "delivery_date_local", "source_folder", "source_file_path", "bronze_ingested_at"]

# --- Document schema for from_json (backticks: from/to are reserved words) --
PAYLOAD_SCHEMA = """
STRUCT<
  queryDataDto: STRUCT<
    dataItemExchangeList: ARRAY<
      STRUCT<
        dataItemCode: STRING,
        data: ARRAY<
          STRUCT<
            timeInterval: STRUCT<`from`: STRING, `to`: STRING>,
            businessDimensionMap: MAP<STRING, STRING>,
            instanceAttributeMap: MAP<STRING, STRING>,
            curveData: STRUCT<
              curveType: STRING,
              periodList: ARRAY<
                STRUCT<
                  timeInterval: STRUCT<`from`: STRING, `to`: STRING>,
                  resolution: STRING,
                  pointMap: MAP<STRING, ARRAY<DOUBLE>>
                >
              >
            >
          >
        >
      >
    >
  >
>
"""

def build_bronze(df):
    """payload VARIANT -> one row per zone / interval / point."""
    d = (
        df.select(
            F.col("payload").cast("string").alias("json_str"),
            F.col("source_folder"),
            F.col("file_path").alias("source_file_path"),
        )
        .withColumn("doc", F.from_json(
            F.regexp_replace(F.col("json_str"), "^\ufeff", ""),  # strip UTF-8 BOM
            PAYLOAD_SCHEMA))
        .select("source_folder", "source_file_path", "doc",
                F.explode("doc.queryDataDto.dataItemExchangeList").alias("item"))
        .select("source_folder", "source_file_path",
                F.col("item.dataItemCode").alias("data_item_code"),
                F.explode("item.data").alias("row"))
        .select("source_folder", "source_file_path", "data_item_code",
                F.col("row.businessDimensionMap").alias("bd"),
                F.col("row.instanceAttributeMap").alias("ia"),
                F.col("row.curveData.curveType").alias("curve_type"),
                F.explode("row.curveData.periodList").alias("period"))
        .select("source_folder", "source_file_path", "data_item_code", "bd", "ia", "curve_type",
                F.to_timestamp(F.col("period.timeInterval.from")).alias("period_from"),
                F.col("period.resolution").alias("resolution"),
                F.explode(F.map_entries(F.col("period.pointMap"))).alias("pt"))
        .withColumn("point_key", F.col("pt.key"))
        .withColumn("point_val", F.col("pt.value"))
        .drop("pt")
    )

    res_secs = (F.when(F.col("resolution") == "PT15M", F.lit(900))
                 .when(F.col("resolution") == "PT60M", F.lit(3600)))
    idx = F.col("point_key").cast("int")
    start = F.timestamp_seconds(F.unix_timestamp(F.col("period_from")) + idx * res_secs)
    end   = F.timestamp_seconds(F.unix_timestamp(start) + res_secs)

    zone = (F.when(F.col("bd")["IN_AREA"] == "10Y1001A1001A44P", "SE1")
              .when(F.col("bd")["IN_AREA"] == "10Y1001A1001A45N", "SE2")
              .when(F.col("bd")["IN_AREA"] == "10Y1001A1001A46L", "SE3")
              .when(F.col("bd")["IN_AREA"] == "10Y1001A1001A47J", "SE4")
              .otherwise(F.col("bd")["IN_AREA"]))

    return d.select(
        zone.alias("bidding_zone"),
        F.col("bd")["IN_AREA"].alias("area_eic"),
        F.col("bd")["CONTRACT_TYPE"].alias("contract_type"),
        F.col("bd")["CLASSIFICATION_SEQUENCE"].alias("classification_sequence"),
        F.col("data_item_code"),
        F.col("ia")["CURRENCY"].alias("currency"),
        F.col("ia")["DATA_PROVIDER_ORGANIZATION"].alias("data_provider_org"),
        F.col("curve_type"),
        F.col("resolution"),
        start.alias("interval_start_utc"),
        end.alias("interval_end_utc"),
        idx.alias("point_index"),
        F.col("point_val")[0].alias("price_eur_mwh"),
        F.to_date(F.from_utc_timestamp(start, "Europe/Stockholm")).alias("delivery_date_local"),
        F.col("source_folder"),
        F.col("source_file_path"),
        F.current_timestamp().alias("bronze_ingested_at"),
    ).where(idx.isNotNull() & F.col("resolution").isin("PT15M", "PT60M"))

def merge_bronze(batch_df, batch_id):
    n_raw = batch_df.count()
    staged = build_bronze(batch_df).cache()

    if n_raw > 0 and staged.isEmpty():
        staged.unpersist()
        raise RuntimeError(f"batch {batch_id}: {n_raw} raw rows but 0 bronze rows - parse failure")

    if staged.isEmpty():
        staged.unpersist()
        return

    sel = ", ".join(COLS)
    staged.createOrReplaceTempView("stg")
    spark.sql(f"""
      MERGE INTO {BRONZE_TABLE} t
      USING (
        SELECT {sel}
        FROM (
          SELECT *, ROW_NUMBER() OVER (
            PARTITION BY area_eic, interval_start_utc
            ORDER BY source_file_path DESC, source_folder DESC
          ) AS rn
          FROM stg
        )
        WHERE rn = 1
      ) s
      ON  t.area_eic = s.area_eic
      AND t.interval_start_utc = s.interval_start_utc
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"batch {batch_id}: raw={n_raw}, staged={staged.count()}")
    staged.unpersist()

# --- Run: incremental via checkpoint ----------------------------------------
query = (
    spark.readStream.format("delta")
    .table(RAW_TABLE)
    .writeStream
    .foreachBatch(merge_bronze)
    .option("checkpointLocation", CKPT)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

# --- Bronze DQ ---------------------------------------------------------------
dups = spark.sql(f"""
  SELECT count(*) AS n FROM (
    SELECT 1 FROM {BRONZE_TABLE}
    GROUP BY area_eic, interval_start_utc
    HAVING count(*) > 1)
""").collect()[0]["n"]
assert dups == 0, f"{dups} duplicate keys in bronze"

spark.sql(f"""
  SELECT min(delivery_date_local) AS first_day,
         max(delivery_date_local) AS last_day,
         count(*)                 AS rows_total,
         count(DISTINCT bidding_zone) AS zones,
         min(price_eur_mwh)       AS min_price,
         max(price_eur_mwh)       AS max_price
  FROM {BRONZE_TABLE}
""").show()

spark.sql(f"""
  SELECT delivery_date_local, count(*) AS pts
  FROM {BRONZE_TABLE}
  GROUP BY 1 ORDER BY 1 DESC LIMIT 7
""").show()